# Chaos temporel en convection de Rayleigh-Bénard — Partie II (bonus)
## Faire une simulation scientifique correctement

Ce notebook suit l'énoncé de la partie II (`projetconvection_partie2.pdf`).
Les **explications** et les **questions** alternent : lisez une explication,
répondez à la question qui suit (dans la cellule de code, puis par écrit),
puis passez à la suite. Exécutez les cellules **dans l'ordre**.

**Conventions.** Une trajectoire `y_res` est un tableau de forme `(3, Nt)` :
lignes $X$, $Y$, $Z$, colonne $n$ à l'instant $t_n = n h$, avec
`h = tmax / (Nt-1)`. Les fonctions fournies sont dans les fichiers du dossier
(`lorenz.py`, `integrateurs.py`, `analyse.py`, `traces_partie2.py`), décrits
dans l'annexe C de l'énoncé. Pour tout ce qui existe déjà dans NumPy, SciPy
ou Matplotlib (normes, pentes, maxima, moyennes, enregistrement des figures),
on utilise directement ces bibliothèques.

**Règle pour les figures.** Chaque figure indique la méthode utilisée (RK4,
DOP853, Euler explicite...), le pas de temps $h$ et les paramètres ($r$, $T$...).

In [ ]:
# Ménage et imports
import time
import cProfile
import pstats
import io
import os
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.optimize import brentq
from scipy.signal import find_peaks

from lorenz import (systeme_dynamique, jacobienne_lorenz, points_fixes,
                    systeme_conservatif, jacobienne_conservatif,
                    invariants_conservatif, oscillateur_harmonique,
                    jacobienne_oscillateur)
from integrateurs import (SCHEMAS, NUMBA_DISPONIBLE, integration_Euler,
                          integration_RK4, integration_Euler_implicite,
                          integration_temporelle_schema,
                          integration_temporelle_python, integration_temporelle,
                          integration_temporelle_vectorisee,
                          integration_temporelle_numba,
                          integration_temporelle_flottant)
from analyse import (solution_reference, charge_reference_json,
                     reference_aux_instants, maxima_Z_parabole,
                     maxima_Z_hermite, correlation_glissante,
                     points_sur_attracteur, fonction_stabilite,
                     valeurs_propres_C_plus, seuil_hopf_numerique,
                     taux_enveloppe, balayage_quasi_statique)
from traces_partie2 import (trace_convergence, trace_domaines_stabilite,
                            trace_facteurs_amplification, trace_enveloppe,
                            style_schema, L_STABLE)

plt.close("all")
# Dossier des figures (enregistrées avec fig.savefig)
DOSSIER_FIGURES = Path("figures_partie2")
DOSSIER_FIGURES.mkdir(exist_ok=True)

# Définition des paramètres
Pr = 10
b = 8/3
print("numba disponible :", NUMBA_DISPONIBLE)

---
# 2. Méthodes d'intégration temporelle : ce qu'il faut savoir

### 2.1 Schémas explicites et implicites

Un schéma à un pas remplace $\dot{\mathbf y} = \mathbf f(\mathbf y)$ par une
itération $\mathbf y_{n+1} = \Phi_h(\mathbf y_n)$. Il est **explicite** si
$\mathbf y_{n+1}$ se calcule directement à partir de $\mathbf y_n$ (Euler
explicite, Heun, RK4). Il est **implicite** si $\mathbf y_{n+1}$ apparaît des
deux côtés de l'égalité : pour Euler implicite,
$\mathbf y_{n+1} = \mathbf y_n + h\,\mathbf f(\mathbf y_{n+1})$. Il faut alors
résoudre à chaque pas un système non linéaire (ici de taille 3) : le dossier
utilise `scipy.optimize.fsolve`, à qui l'on fournit la jacobienne de $\mathbf f$
pour accélérer la résolution.

Le fichier `integrateurs.py` fournit, pour chacun des neuf schémas de l'annexe A,
une fonction qui calcule **un pas** :
* schémas explicites : `ynp1 = integration_RK4(f, yn, t, h)` (de même
  `integration_Euler`, `integration_RK2`) ;
* schémas implicites : `ynp1, nfev = integration_Euler_implicite(f, yn, t, h, jac)`,
  où `jac(t, y)` est la matrice jacobienne de $\mathbf f$ et `nfev` le nombre
  d'appels à la fonction pendant la résolution ;

ainsi que `integration_temporelle_schema(nom_schema, f, y0, Nt, h, jac)`, qui
intègre sur $N_t$ instants avec le schéma choisi (liste des noms : `SCHEMAS`).

**Et SciPy ?** `scipy.integrate.solve_ivp` propose six méthodes, toutes à pas
**adaptatif** (le pas est choisi automatiquement pour respecter une tolérance) :
`RK23`, `RK45`, `DOP853` (explicites), `Radau`, `BDF`, `LSODA`. `BDF` y est une
méthode d'ordre variable (1 à 5) qui contient la formule BDF2, mais on ne
peut imposer ni l'ordre ni le pas. Or, pour mesurer un ordre de convergence,
il faut maîtriser $h$ : c'est pourquoi les schémas à pas fixe sont fournis.

**Question 2.1.** Pour $r = 28$, $\mathbf y_0 = (1, 1, 1)$ et $h = 0.01$,
faites **un pas** d'Euler explicite, d'Euler implicite et de RK4. Comparez-les
à la **solution de référence** à $t = h$, calculée par `solve_ivp` avec la
méthode `DOP853` et des tolérances $10^{-13}$ (fonction `solution_reference`).
Mesurez l'**erreur après un pas** (norme euclidienne de la différence avec la
référence). Combien d'appels à $\mathbf f$ le pas implicite a-t-il demandé ?
Implicite veut-il dire plus précis ?

In [ ]:
# Indication : f = lambda t, y: systeme_dynamique(t, y, Pr, r, b), et de même
# jac = lambda t, y: jacobienne_lorenz(t, y, Pr, r, b)
r = 28
h = 0.01
y0 = np.array([1.0, 1.0, 1.0])
# À compléter : écrivez votre code ici

### 2.2 Consistance, ordre et convergence

L'**erreur locale** (ou erreur de consistance) est l'erreur commise en **un
seul pas** en partant de la solution exacte. Un schéma est **consistant** si
l'erreur locale divisée par $h$ tend vers 0 quand $h \to 0$. Il est **d'ordre
$p$** si l'erreur locale est en $O(h^{p+1})$. L'**erreur globale**
$\mathbf e_n = \mathbf y_n - \mathbf y(t_n)$ accumule les erreurs locales des
$T/h$ pas : pour un schéma stable d'ordre $p$, sur $[0, T]$ fixé,
$\max_n \|\mathbf e_n\| \le C h^p$. En échelle log-log, la courbe d'erreur
globale en fonction de $h$ est donc une droite de **pente théorique $p$**.
Les ordres des schémas sont donnés dans le tableau de l'annexe A.

Attention : cet ordre concerne l'**état** $\mathbf y(t)$. Une grandeur
**calculée à partir** de la trajectoire peut converger avec un autre ordre :
un maximum détecté parmi les échantillons converge en $O(h^2)$ quel que soit
le schéma (section 4.6), et un seuil d'instabilité peut converger en $O(h^{p+1})$
(section 5.3). Il faut donc toujours mesurer l'ordre **de la grandeur qui
nous intéresse**.

### 2.3 Stabilité : l'équation test

On applique le schéma à l'**équation test** $\dot y = \lambda y$ ($\lambda$
complexe) : un pas donne $y_{n+1} = R(h\lambda)\, y_n$, où $R$ est la
**fonction de stabilité** du schéma. Pour Euler explicite,
$y_{n+1} = (1 + h\lambda) y_n$ donc $R(z) = 1 + z$ ; pour Euler implicite,
$y_{n+1} = y_n + h\lambda y_{n+1}$ donc $R(z) = 1/(1 - z)$. Comme
$y_n = R(h\lambda)^n y_0$, la solution numérique est amortie si $|R| < 1$,
explose si $|R| > 1$, garde son amplitude si $|R| = 1$. Le **taux de
croissance numérique** vaut $\sigma_{num} = \frac1h \ln|R(h\lambda)|$, à
comparer au taux exact $\mathrm{Re}\,\lambda$.

Pourquoi cette équation ? Près d'un point fixe, le système linéarisé se
décompose sur les vecteurs propres de la jacobienne, et chaque composante
obéit à une équation test, $\lambda$ étant une valeur propre.

**Question 2.3.** Vérifiez numériquement les fonctions de stabilité. Un nombre
complexe $y = u + iv$ obéissant à $\dot y = \lambda y$, avec $\lambda = a + i\omega$,
est équivalent au système réel $\dot u = a u - \omega v$, $\dot v = \omega u + a v$.
Pour $h\lambda = z = -0.5 + 0.5i$ ($h = 1$), faites un pas de chaque schéma
depuis $(u, v) = (1, 0)$ : le résultat $u_1 + i v_1$ vaut $R(z)$. Comparez à
`fonction_stabilite(nom_schema, z)` (BDF2, schéma à deux pas, est exclu).

In [ ]:
# Indication : un pas s'obtient avec integration_temporelle_schema et Nt = 2 ;
# jacobienne de ce système linéaire : [[a, -w], [w, a]]
z = -0.5 + 0.5j
h = 1.0
a, w = z.real / h, z.imag / h
# À compléter : écrivez votre code ici

### 2.4 A-stabilité, L-stabilité, conservation

Le **domaine de stabilité** est l'ensemble des $z$ tels que $|R(z)| \le 1$ :
le schéma est stable pour un pas $h$ si $h\lambda$ est dans ce domaine pour
toutes les valeurs propres $\lambda$ du problème.

* **A-stable** : le domaine contient tout le demi-plan $\mathrm{Re}\,z \le 0$.
  Tout mode amorti dans la réalité reste amorti dans la simulation, **quel que
  soit $h$**. Aucun schéma explicite n'est A-stable.
* **L-stable** : A-stable et, de plus, $|R(z)| \to 0$ quand $h\lambda \to -\infty$.
  Un mode très rapidement amorti est éliminé en un pas. Pour Crank-Nicolson et
  le point milieu, $R(z) \to -1$ : un tel mode change de signe à chaque pas
  sans décroître.
* **Conservation** : sur l'axe imaginaire ($\lambda = i\omega$, oscillation non
  amortie), $|R(i\omega h)| = 1$ signifie que l'amplitude des oscillations
  (l'énergie) est conservée ; $< 1$ : dissipée ; $> 1$ : injectée.

Ces propriétés sont binaires (« oui » ou « non »), alors que l'écart peut être
minuscule : on regarde donc aussi les **valeurs** de $|R(iy)|$ et de $|R(-x)|$.

**Lire un domaine de stabilité.** La zone stable est **coloriée en bleu**.
Pour un schéma explicite, c'est une zone bornée (l'intérieur de la courbe).
Pour un schéma implicite, c'est une zone non bornée : pour Euler implicite,
c'est tout l'**extérieur** du disque de centre 1 et de rayon 1, qui est situé
dans le demi-plan droit. La courbe $|R| = 1$ seule ne suffit donc pas : il faut
savoir de quel côté se trouve la zone stable.

**Question 2.4.a.** Tracez les domaines de stabilité (`trace_domaines_stabilite`)
et les facteurs d'amplification (`trace_facteurs_amplification`) des neuf
schémas. Complétez le tableau ci-dessous (double-cliquez sur la cellule pour
l'éditer ; le même tableau est fourni dans `tableau_recapitulatif.csv`).

| Schéma | Ordre | A-stable | L-stable | $\lvert R(i)\rvert$ | $\lim \lvert R(-x)\rvert$ | Conserve l'énergie (oscillateur) |
|---|---|---|---|---|---|---|
| Euler explicite | 1 | | | | | |
| Heun (RK2) | 2 | | | | | |
| RK4 | 4 | | | | | |
| Euler implicite | 1 | | | | | |
| Crank-Nicolson | 2 | | | | | |
| Point milieu | 2 | | | | | |
| BDF2 | 2 | | | | | |
| SDIRK2 | 2 | | | | | |
| TR-BDF2 | 2 | | | | | |

**Question 2.4.b.** Heun et RK4 ne sont stables que dans une petite zone autour
de 0 (par exemple, RK4 est instable dès que $\mathrm{Re}\,z < -2.79$ sur l'axe
réel). Que cela signifie-t-il concrètement pour le pas $h$ ? Pour quel genre de
problème est-ce gênant, et pour quel genre de problème est-ce sans importance ?

In [ ]:
noms_schemas = list(SCHEMAS)
# À compléter : écrivez votre code ici

### 2.5 Raideur et condition CFL

La condition **CFL** (Courant, Friedrichs, Lewy, 1928) est née pour les
équations aux dérivées partielles : pour le transport $\partial_t u + c\,\partial_x u = 0$
discrétisé avec un pas $\Delta x$ et un schéma explicite, la stabilité impose
$|c|\Delta t / \Delta x \le 1$ (l'information ne doit pas parcourir plus d'une
maille par pas) ; pour la chaleur $\partial_t u = \kappa \partial_{xx} u$, il faut
$\kappa \Delta t/\Delta x^2 \le 1/2$.

C'est un cas particulier du domaine de stabilité : discrétiser l'EDP en
espace donne un système $\dot{\mathbf y} = A\mathbf y$ dont les valeurs
propres sont de l'ordre de $|c|/\Delta x$ ou $\kappa/\Delta x^2$. Un schéma
explicite n'est stable que si $h\lambda_i$ est dans son domaine (borné) pour
toutes les valeurs propres, d'où
$$ h \le \frac{C}{\max_i |\lambda_i|}, $$
avec $C = 2$ pour Euler explicite et $C \approx 2.785$ pour RK4 si les
$\lambda_i$ sont réels négatifs. Un problème est **raide** lorsque c'est la
stabilité, et non la précision, qui limite le pas d'un schéma explicite.

**Question 2.5.** Pour $\dot y = -50\, y$, prédisez le pas maximal d'Euler
explicite et de RK4. Vérifiez-le : intégrez jusqu'à $t = 2$ avec un pas égal à
0.98 et à 1.02 fois la prédiction, et tracez $|y(t)|$ en échelle log.

In [ ]:
lam = -50.0
f_lineaire = lambda t, y: lam * y
# À compléter : écrivez votre code ici

### 2.6 Règles de base : quelle méthode pour quel problème ?

* **Non raide, solution régulière, précision recherchée** : explicite d'ordre
  élevé (RK4 à pas fixe ; `DOP853` ou `RK45` de `solve_ivp` à pas adaptatif).
  C'est le cas de Lorenz pour $P_r = 10$.
* **Raide** : implicite L-stable (à pas fixe : Euler implicite, BDF2, SDIRK2,
  TR-BDF2 ; à pas adaptatif : `Radau` ou `BDF` de `solve_ivp`, ou `LSODA`, qui
  détecte la raideur). Éviter Crank-Nicolson et le point milieu, A-stables mais
  non L-stables (oscillations parasites, section 6.2).
* **Conservatif** (invariant à préserver sur des temps longs) : point milieu
  (ou Crank-Nicolson). Éviter Euler implicite (dissipe) et explicite (amplifie).
* **Seuil d'une instabilité oscillante** : éviter les schémas d'ordre bas
  qui dissipent ou amplifient (ils déplacent le seuil, section 5.3).
* **Chaotique** : la trajectoire n'est calculable que jusqu'à un horizon fini ;
  au-delà, seules les statistiques ont un sens (sections 4.4 et 4.5).
* **Mesure d'un ordre de convergence** : pas fixe ; **production** : pas
  adaptatif. Toujours vérifier qu'un résultat change peu quand on divise $h$
  (ou la tolérance) par 2.

### 2.7 Expérience : l'oscillateur harmonique

L'oscillateur $\ddot x + x = 0$ s'écrit $\dot x = v$, $\dot v = -x$ ; ses
valeurs propres sont $\pm i$ et son énergie $E = (x^2+v^2)/2$ est constante.
La fonction `taux_enveloppe(signal, ts)` repère les maxima successifs de
$|x(t)|$ et ajuste $\ln(\max_k) = \sigma t_k + c$ : $\sigma$ est le taux de
croissance de l'amplitude.

**Question 2.7.a.** Intégrez l'oscillateur depuis $(x_0, v_0) = (1, 0)$ pendant
100 périodes ($T = 200\pi$) avec $h = 0.1$, pour les neuf schémas. Tracez
l'amplitude (maxima successifs de $|x|$) et l'écart relatif d'énergie
$|E(t)/E(0) - 1|$. Mesurez $\sigma$ et comparez à $\frac1h \ln|R(ih)|$. Quels
schémas conservent l'énergie, la dissipent, l'amplifient ?

**Question 2.7.b.** Les courbes de Crank-Nicolson et du point milieu sont
parfaitement superposées. Pourquoi ? (Écrivez les deux schémas pour
$\mathbf f(\mathbf y) = A\mathbf y$, avec $A$ une matrice.) Que reste-t-il comme
différence entre les deux calculs ?

In [ ]:
h = 0.1
Nt = int(200*np.pi / h) + 1
ts = np.arange(Nt) * h
# À compléter : écrivez votre code ici

---
# 3. Performance : mesurer avant d'optimiser

### 3.1 Mesurer un temps de calcul

`time.perf_counter()` renvoie la valeur (en secondes) d'une horloge de haute
résolution ; seule la **différence** entre deux appels a un sens :
```python
debut = time.perf_counter()
...  # calcul
duree = time.perf_counter() - debut
```
D'autres programmes tournent en même temps sur l'ordinateur : une mesure
isolée est bruitée. On répète la mesure et on garde le **minimum**.

Le fichier `integrateurs.py` fournit quatre versions du **même** calcul
(RK4 pour le système de Lorenz), qui renvoient des trajectoires identiques au
bit près car elles font les mêmes opérations dans le même ordre :
* `integration_temporelle_python` : **pur Python**. L'état est formé de trois
  flottants $X$, $Y$, $Z$ ; les quatre pentes sont écrites composante par
  composante ; la trajectoire est stockée dans trois listes ;
* `integration_temporelle` : **NumPy à boucles**. L'état est un tableau de
  taille 3 ; à chaque pas, `integration_RK4` appelle quatre fois
  `systeme_dynamique`, qui crée un tableau avec `np.array` ;
* `integration_temporelle_vectorisee` : **NumPy vectorisé**. Même boucle en
  temps, mais l'état est de forme `(3, M)` pour $M$ valeurs de $r$ à la fois
  (question 3.2) ;
* `integration_temporelle_numba` : le calcul de la version NumPy à boucles,
  **compilé** par `numba.njit` (question 3.1.c). C'est la version utilisée
  pour tous les calculs longs de ce notebook.

**Question 3.1.a.** Mesurez le temps par pas des versions pur Python, NumPy à
boucles et Numba pour $N_t = 20001$, $h = 0.01$, $r = 28$ (minimum de 3 mesures).
Vérifiez que les trois trajectoires sont identiques. Tracez les temps
(`plt.bar`, échelle log).

In [ ]:
Nt = 20001
h = 0.01
r = 28
y0 = [1, 1, 1]


def chronometre(fonction, *arguments, repetitions=3):
    # Temps minimal (en s) de fonction(*arguments) sur plusieurs répétitions
    durees = []
    for _ in range(repetitions):
        debut = time.perf_counter()
        fonction(*arguments)
        durees.append(time.perf_counter() - debut)
    return min(durees)


integration_temporelle_numba(y0, 10, Pr, r, b, h)   # compilation (voir 3.1.c)
# À compléter : écrivez votre code ici

**Question 3.1.b.** La boucle NumPy est la plus lente, alors que NumPy est
réputé rapide. Pourquoi ? *Indication :* chaque opération NumPy coûte un temps
fixe de l'ordre de la microseconde, quelle que soit la taille du tableau ;
combien d'opérations sur des tableaux de 3 nombres un pas de RK4 enchaîne-t-il ?

**Numba et le premier appel.** Numba traduit une fonction Python en code
machine (compilation *just-in-time*) au moment de son **premier** appel :
analyse du code, déduction des types des variables, compilation, soit 1 à 5 s.
Les appels suivants (avec des arguments de même type) utilisent directement
le code compilé. Mesurer le premier appel reviendrait à chronométrer le
compilateur, et non le calcul : c'est pourquoi la cellule précédente appelle
une fois `integration_temporelle_numba` **avant** de chronométrer.

**Question 3.1.c.** Redémarrez le noyau (menu *Redémarrer*), exécutez la cellule
des imports puis la cellule suivante : comparez les durées du premier et du
second appel de la version Numba (10 pas).

In [ ]:
# À compléter : écrivez votre code ici

**Profiler.** Le module `cProfile` (bibliothèque standard de Python)
enregistre, pour chaque fonction appelée, le nombre d'appels (`ncalls`), le
temps passé dans la fonction elle-même (`tottime`) et le temps cumulé avec
ses sous-fonctions (`cumtime`) : il dit **où** le temps est dépensé. On peut
aussi profiler un script entier depuis un terminal :
`python -m cProfile -s tottime mon_script.py`.

**Question 3.1.d.** Profilez la boucle NumPy (`integration_temporelle`) pour
$N_t = 20001$. Combien de fois `systeme_dynamique` est-elle appelée, et
pourquoi ? Quelle fraction du temps total représente-t-elle ?

In [ ]:
# À compléter : écrivez votre code ici

### 3.2 Vectoriser sur le paramètre r

Dans un diagramme de bifurcation, les intégrations pour différentes valeurs
de $r$ sont indépendantes. `integration_temporelle_vectorisee` les fait
**simultanément** : l'état est de forme `(3, M)` et, grâce au *broadcasting*
de NumPy, l'expression `-X*Z + r*X - Y` de `systeme_dynamique` s'applique
composante par composante aux $M$ systèmes (avec `r` un tableau de taille $M$).
Le coût fixe de chaque opération est alors partagé entre les $M$ valeurs de $r$.

**Question 3.2.** Pour $M = 100$ valeurs de $r \in [30, 350]$, $N_t = 5001$ et
$h = 0.005$, comparez le temps de calcul d'une boucle sur $r$ avec chacune des
trois versions de la question 3.1 et celui de la version NumPy vectorisée.
Vérifiez que les états finaux sont identiques. Refaites la mesure de la
version vectorisée pour $M = 1000$ : que constatez-vous ?

In [ ]:
Nt = 5001
h = 0.005
rs = np.linspace(30, 350, 100)
# À compléter : écrivez votre code ici

### 3.3 Mémoire et calcul « au vol »

La mémoire occupée par un tableau vaut (nombre de nombres) × (octets par
nombre) : 8 octets en double précision (annexe B). `y_res.nbytes` la donne
directement.

**Question 3.3.a.** Un diagramme de bifurcation calculé avec
`integration_temporelle_vectorisee` pour $M$ valeurs de $r$ et $N_t$ instants
stocke un tableau de forme `(M, 3, Nt)`. Écrivez la formule de sa taille en
octets, puis calculez-la (sans faire la simulation) pour $M = 321$,
$N_t = 2^{16}$, puis $M = 1000$, $N_t = 10^6$. Comparez à la mémoire vive d'un
ordinateur portable (8 à 16 Go). Vérifiez la formule sur un petit cas avec `.nbytes`.

In [ ]:
# À compléter : écrivez votre code ici

**Calcul au vol.** On appelle ainsi un calcul effectué **pendant**
l'intégration, sans stocker la trajectoire : dans la boucle en temps, on ne
garde en mémoire que les trois dernières valeurs $Z_{n-1}$, $Z_n$, $Z_{n+1}$ ;
si $Z_{n-1} < Z_n > Z_{n+1}$, on enregistre $Z_n$ (un maximum), puis on décale
les trois valeurs. Seule la liste des maxima est conservée.

**Question 3.3.b.** Écrivez une fonction `maxima_Z_au_vol(y0, Nt, Pr, r, b, h)`
(RK4, compilée avec `numba.njit`) qui renvoie les maxima de $Z$ calculés au
vol. Pour 21 valeurs de $r \in [30, 350]$, $t_{max} = 100$, $N_t = 2^{16}$,
$\mathbf y_0 = (1, 0, 0)$, vérifiez qu'elle donne les mêmes 125 derniers maxima
que la méthode avec stockage (`integration_temporelle_vectorisee` puis
`scipy.signal.find_peaks` sur $Z$). Enregistrez les maxima sur le disque avec
`np.save` et comparez la taille du fichier (`os.path.getsize`) à celle qu'aurait
le fichier des trajectoires complètes.

In [ ]:
# À compléter : écrivez votre code ici

---
# 4. Erreurs et convergence en pas de temps

### 4.1 Solution de référence et normes d'erreur

Le système de Lorenz n'a pas de solution analytique : on compare le schéma à
une **solution de référence** calculée avec une méthode beaucoup plus précise,
`DOP853` (Runge-Kutta explicite d'ordre 8, pas adaptatif) de `solve_ivp`, avec
des tolérances $10^{-13}$. Trois références sont fournies dans `data` (fichiers
texte `.json`, lus par `charge_reference_json`) :
* `"r10"` : $r = 10$, $\mathbf y_0 = (1,1,1)$, $t \in [0, 2]$ ;
* `"r10_precise"` : la même, avec des tolérances encore plus sévères (section 4.2) ;
* `"r28"` : $r = 28$, $\mathbf y_0 = (1,1,1)$, $t \in [0, 30]$.

`reference_aux_instants(ref, ts)` renvoie leurs valeurs (tableau `(3, len(ts))`)
aux instants voulus ; `solution_reference(y0, tmax, dt_sortie, Pr, r, b)` en
calcule une nouvelle. Pour **vérifier** une référence, on la compare à une
seconde référence de tolérances moins sévères ($10^{-11}$) : si l'écart est très
petit devant les erreurs à mesurer, la référence la plus précise est fiable.

**Question 4.1.a.** Pour $r = 10$ ($t \in [0, 2]$) et $r = 28$ ($t \in [0, 30]$),
calculez la solution avec des tolérances $10^{-11}$ et tracez, en fonction du
temps, la norme euclidienne de son écart avec la référence fournie. Mesurez
l'écart maximal dans les deux cas. Pour $r = 10$, l'écart reste-t-il contenu ?
Et pour $r = 28$ ? Que peut-on en conclure sur le régime chaotique ?

In [ ]:
ref_r10 = charge_reference_json("r10")
ref_r28 = charge_reference_json("r28")
# À compléter : écrivez votre code ici

**Normes d'erreur.** À chaque instant $t_n$, l'**erreur instantanée** est la
norme euclidienne $e_n = \|\mathbf y_n - \mathbf y_{ref}(t_n)\|$ : c'est une
**fonction du temps**, que l'on résume par un nombre de plusieurs façons.
Avec `e` le tableau des $e_n$ (`e = np.linalg.norm(y_res - y_ref, axis=0)`) :
* **erreur finale** : $E_T = e_N$ (`e[-1]`) ;
* **norme infinie** (plus grande erreur instantanée sur $[0, T]$) :
  $E_\infty = \max_n e_n$ (`np.linalg.norm(e, np.inf)`) ;
* **norme $L^2$ discrète** : $E_2 = (h \sum_n e_n^2)^{1/2}$ (`np.sqrt(h) * np.linalg.norm(e, 2)`) ;
* **norme $L^1$ discrète** : $E_1 = h \sum_n e_n$ (`h * np.linalg.norm(e, 1)`).

$E_2/\sqrt T$ et $E_1/T$ sont des erreurs **moyennes** sur $[0, T]$ (moyenne
quadratique et moyenne simple des $e_n$). On a $E_1 \le \sqrt T E_2 \le T E_\infty$.

**Question 4.1.b** (sur papier). Laquelle de ces erreurs utiliser (i) si
l'on a besoin que la trajectoire soit précise **à tous les instants** (par
exemple pour suivre une particule) ; (ii) si l'on veut une idée de l'erreur
**typique** sur $[0, T]$ ; (iii) si l'on ne s'intéresse qu'à la **moyenne en
temps** d'une grandeur, par exemple $\langle Z\rangle = \frac1T\int_0^T Z\,dt$ ?
Pour (iii), montrez que $|\langle Z\rangle_h - \langle Z\rangle_{ref}| \le E_1/T$.
*Indication :* $|\int g| \le \int |g|$ et $|Z_h - Z_{ref}| \le e$.

### 4.2 Ordre de convergence en régime non chaotique

On intègre $r = 10$, $\mathbf y_0 = (1,1,1)$ jusqu'au **temps final $T = 2$**
(pendant le transitoire, avant que la solution n'ait rejoint le point fixe),
avec $h = 2^{-k}$. Les pentes théoriques sont les ordres de l'annexe A.

Le **régime pré-asymptotique** est la plage des grands $h$ où l'erreur n'est
pas encore proportionnelle à $h^p$ (les termes d'ordre supérieur ne sont pas
négligeables) : on mesure la pente en dehors (arguments `hmin`, `hmax` de
`trace_convergence`, qui grise la zone exclue).

**Question 4.2.a.** Tracez la norme infinie $E_\infty$ en fonction de $h$ pour
les neuf schémas ($k = 3, \dots, 12$ ; jusqu'à 14 pour RK4, avec
`integration_temporelle_numba`). Mesurez les pentes. Où est le régime
pré-asymptotique de chaque schéma ? Que se passe-t-il pour Euler explicite
avec $h = 1/8$ ?

In [ ]:
T = 2
r = 10
f = lambda t, y: systeme_dynamique(t, y, Pr, r, b)
jac = lambda t, y: jacobienne_lorenz(t, y, Pr, r, b)
# Ordres théoriques (annexe A)
ordres = {"Euler explicite": 1, "Heun (RK2)": 2, "RK4": 4, "Euler implicite": 1,
          "Crank-Nicolson": 2, "Point milieu": 2, "BDF2": 2, "SDIRK2": 2, "TR-BDF2": 2}
# À compléter : écrivez votre code ici

**Plancher d'erreur.** Pour RK4, l'erreur cesse de décroître vers $10^{-12}$ :
c'est un **plancher d'erreur** (*error floor*, ou *round-off plateau* quand il
est dû à l'arrondi). Deux causes possibles : la précision limitée de la
**référence**, et l'**erreur d'arrondi** (chaque opération sur des flottants est
arrondie avec une erreur relative de l'ordre de la précision machine
$\varepsilon$, annexe B).

**Question 4.2.b.** (1) Refaites la courbe de RK4 avec la référence plus
précise `"r10_precise"` : le plancher bouge-t-il ? (2) Refaites-la en **simple
précision** avec `integration_temporelle_flottant(..., np.float32)` ($k = 4, \dots, 14$).
Mesurez la pente de chaque courbe **sur la plage où elle décroît**. Pourquoi
l'erreur en simple précision **augmente**-t-elle quand $h$ diminue sous un
certain seuil ? *Indication :* l'erreur d'arrondi à chaque pas est de l'ordre
de $\varepsilon \|\mathbf y\|$, et il y a $T/h$ pas.

In [ ]:
ref_r10_precise = charge_reference_json("r10_precise")
# À compléter : écrivez votre code ici

**Le piège du point fixe.** Un **point fixe** (équilibre) $\mathbf y^*$ vérifie
$\mathbf f(\mathbf y^*) = 0$ : la solution exacte qui part de $\mathbf y^*$ y
reste. Un schéma de Runge-Kutta le laisse aussi inchangé : si
$\mathbf y_n = \mathbf y^*$, alors $\mathbf k_1 = \mathbf f(\mathbf y^*) = 0$,
$\mathbf k_2 = \mathbf f(\mathbf y^* + \frac h2 \mathbf k_1) = 0$, etc., donc
$\mathbf y_{n+1} = \mathbf y_n$ : **aucune erreur au point fixe**.

**Question 4.2.c.** Pour $r = 10$, toute trajectoire converge vers $C_+$ ou $C_-$.
Mesurez l'erreur finale $E_T$ à $T = 50$ pour Euler explicite et RK4
($h = 0.01$), en prenant comme référence le point fixe atteint. Qu'en
concluez-vous sur le choix de l'instant de mesure ?

In [ ]:
# À compléter : écrivez votre code ici

### 4.3 Diagramme travail-précision

Une méthode d'ordre élevé fait moins de pas, mais chaque pas coûte plus cher.
Pour comparer des méthodes, on trace l'**erreur obtenue en fonction du
travail** : le nombre d'appels à $\mathbf f$ (indépendant de l'ordinateur) et le
temps de calcul. La meilleure méthode, pour une précision donnée, est celle
dont la courbe est la plus à gauche. Pour les méthodes à pas adaptatif de
`solve_ivp`, on fait varier la tolérance (`rtol = atol`) au lieu de $h$ ; le
nombre d'appels est donné par `sol.nfev`.

**Question 4.3.** Pour $r = 10$, $\mathbf y_0 = (1,1,1)$, $T = 2$ (système non
raide), tracez l'**erreur finale** $E_T$ (référence `"r10"`) en fonction du
nombre d'appels à $\mathbf f$, puis du temps de calcul, pour : Euler explicite,
Heun, RK4 et Crank-Nicolson à pas fixe ($h = 2^{-k}$, $k = 4, \dots, 11$, avec
`integration_temporelle_schema`, qui renvoie aussi le nombre d'appels), et
`RK45`, `DOP853`, `Radau`, `LSODA` de `solve_ivp` avec
`rtol = atol` $= 10^{-3}, \dots, 10^{-12}$. Quelle méthode choisir pour une
précision de $10^{-3}$ ? de $10^{-8}$ ? Le schéma implicite est-il intéressant ici ?

In [ ]:
y_T_ref = ref_r10["y_ref"][:, -1]
# À compléter : écrivez votre code ici

### 4.4 Régime chaotique : l'horizon de prédictibilité

En régime chaotique, deux trajectoires très proches s'écartent en moyenne
exponentiellement : $\|\delta\mathbf y(t)\| \approx \|\delta\mathbf y(0)\| e^{\lambda t}$.
Le taux $\lambda$ est l'**exposant de Lyapunov** maximal ; pour $r = 28$,
$\lambda \approx 0.906$. Son inverse $1/\lambda \approx 1.1$ est le **temps de
Lyapunov** : une petite erreur est multipliée par $e$ toutes les 1.1 unités
de temps, par 10 toutes les 2.5 unités. Les erreurs numériques sont amplifiées
de la même façon : au-delà d'un certain temps, la simulation et la solution de
référence (qui a elle-même une erreur) deviennent des trajectoires différentes
de l'attracteur. Les trajectoires restent toutefois sur l'attracteur, qui est
borné ($|X|, |Y| \lesssim 25$, $0 \le Z \lesssim 50$) : leur écart ne peut pas
dépasser sa taille, d'environ 50.

**Question 4.4.a.** Pour $r = 28$, $\mathbf y_0 = (1,1,1)$, $t \in [0, 30]$ et RK4
avec $h = 0.02, 0.01, 0.005, 0.0025$, tracez l'erreur instantanée
$e(t) = \|\mathbf y_h(t) - \mathbf y_{ref}(t)\|$ (référence `"r28"`). Décrivez les
trois phases. À quelle valeur l'erreur sature-t-elle, et pourquoi ? Tracez
aussi la corrélation glissante entre $X_h(t)$ et $X_{ref}(t)$ pour $h = 0.01$
(`correlation_glissante`, fenêtres de durée 1) : le coefficient de corrélation
vaut 1 si les deux signaux varient ensemble, environ 0 s'ils n'ont plus de
lien. À partir de quand la simulation ne suit-elle plus la solution de référence ?

In [ ]:
tmax = 30
r = 28
hs_chaos = [0.02, 0.01, 0.005, 0.0025]
# À compléter : écrivez votre code ici

**Horizon de prédictibilité.** On définit $T_p(h)$ comme le premier instant où
l'erreur instantanée $e(t)$ devient supérieure à $\delta = 1$, soit environ 2 %
de la taille de l'attracteur (`ts[np.argmax(e > 1)]`). Ce seuil est petit devant
la taille de l'attracteur (la simulation est encore proche de la référence)
mais grand devant l'erreur initiale (la croissance exponentielle a eu lieu).
En modélisant $e(t) \approx C h^p e^{\lambda t}$, on obtient
$$ T_p \approx \frac1\lambda \ln\frac{\delta}{C} + \frac{p}{\lambda}\ln\frac1h. $$
Changer $\delta$ ne fait que décaler $T_p$ de $\frac1\lambda\ln\frac{\delta'}{\delta}$.

**Question 4.4.b.** Calculez $T_p(h)$ pour 20 conditions initiales sur
l'attracteur (`points_sur_attracteur(20, Pr, 28, b)`), avec pour chacune une
référence `solution_reference(y0, 25, 2.0**-10, Pr, 28, b)` (0.5 s chacune).
Tracez les 20 valeurs et leur moyenne en fonction de $\ln(1/h)$. Pourquoi
sont-elles dispersées ? Estimez $\lambda$ à partir des deux plus petits pas.
Que gagne-t-on en divisant $h$ par 2, et à quel coût ?

In [ ]:
# À compléter : écrivez votre code ici

**Le mur de l'arithmétique flottante.** Même un schéma parfait commet, à
chaque opération, une erreur d'arrondi relative de l'ordre de la précision
machine $\varepsilon$ (annexe B), qui est ensuite amplifiée comme $e^{\lambda t}$.
En partant d'une erreur initiale $e_0 \approx \varepsilon$, l'erreur atteint
$\delta$ au bout d'un temps
$$ T_p \approx \frac1\lambda \ln\frac{\delta}{\varepsilon}. $$

**Question 4.4.c.** Comparez deux simulations RK4 identiques ($h = 0.001$,
$t \in [0, 25]$), l'une en double précision (`integration_temporelle_numba`),
l'autre en simple précision (`integration_temporelle_flottant`). À quel
instant leur écart dépasse-t-il 1 ? Comparez à la formule ci-dessus avec
$\varepsilon \approx 6\cdot10^{-8}$ (simple précision), $\delta = 1$ et
$\lambda = 0.906$. Que donne la même formule en double précision
($\varepsilon \approx 1.1\cdot10^{-16}$) ?

In [ ]:
# À compléter : écrivez votre code ici

### 4.5 Convergence statistique

Au-delà de l'horizon de prédictibilité, les trajectoires ne convergent plus,
mais les **statistiques** sur l'attracteur (moyennes, histogrammes) peuvent
converger. On étudie la **moyenne temporelle**
$\langle Z\rangle_T = \frac1T\int_{t_0}^{t_0+T} Z\,dt$, calculée par `np.mean`
sur la trajectoire après un transitoire $t_0 = 50$, pour $r = 28$. Elle a deux
erreurs de natures différentes :
* l'**erreur d'échantillonnage**, due à la durée finie $T$ (on n'a observé
  qu'une portion de l'attracteur) : elle change d'une condition initiale à
  l'autre ;
* l'**erreur de discrétisation** (ou biais), due au pas $h$ : elle est la même
  pour toutes les conditions initiales.

On les étudie séparément (a et b), puis ensemble (d).

**Question 4.5.a (erreur d'échantillonnage).** $r = 28$, RK4, $h = 0.01$ (biais
négligeable), 10 conditions initiales. Pour $T = 50, 100, 200, 400$, calculez
l'écart-type des 10 valeurs de $\langle Z\rangle_T$. Vérifiez qu'il décroît comme
$T^{-1/2}$.

In [ ]:
rng = np.random.default_rng(0)
conditions_initiales = [rng.normal(size=3)*5 + np.array([0, 0, 20]) for _ in range(10)]


def moyenne_Z(y0, h, T, schema="RK4", transitoire=50):
    # <Z> sur [transitoire, transitoire + T] : intégration (numba) puis np.mean
    n0 = int(round(transitoire / h))
    y_res = integration_temporelle_numba(y0, n0 + int(round(T / h)) + 1, Pr, 28, b, h, schema)
    return np.mean(y_res[2, n0:])


# À compléter : écrivez votre code ici

**Question 4.5.b (erreur de discrétisation).** $T = 400$, mêmes conditions
initiales, **mêmes pas** pour les deux schémas : $h = 0.01, 0.005, 0.0025, 0.00125, 0.000625$.
Pour Euler explicite et RK4, calculez $\langle Z\rangle$ (moyenne des 10 valeurs ;
incertitude = écart-type divisé par $\sqrt{10}$). Prenez comme valeur de
référence celle de RK4 au plus petit pas, et tracez le **diagramme de
convergence des statistiques** : $|\langle Z\rangle_h - \langle Z\rangle_{ref}|$
en fonction de $h$ pour les deux schémas, avec le niveau de l'incertitude
statistique. Le biais est-il visible pour RK4 ? Avec quel ordre décroît-il pour
Euler explicite ?

In [ ]:
hs_stat = np.array([0.01, 0.005, 0.0025, 0.00125, 0.000625])
# À compléter : écrivez votre code ici

**Comparer des distributions.** La **distance de Wasserstein**
(`scipy.stats.wasserstein_distance`) compare deux distributions de même masse
totale (densités de probabilité, histogrammes normalisés) : c'est le coût
minimal pour transformer l'une en l'autre en **déplaçant** de la masse
(*earth mover's distance*, transport optimal). Contrairement à une différence
point par point, elle est sensible au **décalage** : décaler une distribution
de $a$ donne une distance $|a|$. Elle s'exprime dans l'unité de la variable.

**Question 4.5.c.** La **densité de probabilité de $Z$** sur l'attracteur
indique la fraction du temps que la trajectoire passe à chaque valeur de $Z$.
Estimez-la par un histogramme normalisé (`plt.hist(..., density=True)`) de $Z(t)$,
$t \in [50, 400]$, pour RK4 ($h = 0.01$) avec deux conditions initiales et pour
Euler explicite ($h = 0.005$). Calculez les distances de Wasserstein et
indiquez-les sur la figure.

**Question 4.5.d** (sur papier). L'erreur totale sur $\langle Z\rangle$ est la
somme d'une erreur d'échantillonnage (aléatoire, $\propto T^{-1/2}$) et d'un biais
($\propto h^p$). On suppose ces deux erreurs **indépendantes** : est-ce
raisonnable ? Comment le vérifier avec vos résultats (l'écart-type entre
conditions initiales dépend-il de $h$) ? Pour un budget de calcul donné
(nombre total de pas), comment répartir l'effort entre un petit $h$ et un grand $T$ ?

In [ ]:
from scipy.stats import wasserstein_distance
# À compléter : écrivez votre code ici

### 4.6 Précision des maxima et de l'application de Lorenz

Pour l'application de Lorenz, on détecte les maxima de $Z$ parmi les valeurs
échantillonnées : $Z_n$ est un maximum si $Z_{n-1} < Z_n > Z_{n+1}$
(`scipy.signal.find_peaks` fait exactement cela). Le vrai maximum est en
général atteint **entre** deux instants $t_n$. Deux détecteurs plus précis sont
fournis dans `analyse.py` :
* `maxima_Z_parabole` : parabole passant par l'échantillon maximal et ses deux
  voisins ; on retient son **sommet** (et non plus l'échantillon) ;
* `maxima_Z_hermite` : on utilise aussi $\dot Z = XY - bZ$ ;
  `scipy.interpolate.CubicHermiteSpline` construit sur chaque pas le polynôme
  de degré 3 qui a les bonnes valeurs **et** les bonnes dérivées ; les maxima
  sont les zéros de sa dérivée.

Les ordres attendus sont : 2 pour le détecteur naïf (développement
$Z(t^*+\delta) \approx Z^* - \frac12|\ddot Z|\delta^2$ avec $|\delta| \le h/2$),
3 pour la parabole (erreur d'interpolation d'un polynôme de degré 2), et
l'ordre du schéma (4 pour RK4) pour Hermite.

**Question 4.6.** $r = 28$, $\mathbf y_0 = (1,1,1)$, $t \in [0, 15]$, RK4 avec
$h = 2^{-6}, \dots, 2^{-10}$. Les maxima de référence s'obtiennent avec le
détecteur d'Hermite appliqué à la référence `"r28"`. Tracez l'erreur maximale
sur **tous** les maxima de $[0, 15]$ en fonction de $h$ pour les trois
détecteurs, avec les pentes théoriques. Quelle erreur domine avec le détecteur
naïf : celle du schéma ou celle du détecteur ?

In [ ]:
# À compléter : écrivez votre code ici

---
# 5. Stabilité des points fixes et seuils de transition

### 5.1 Points fixes et analyse linéaire

Un **point fixe** vérifie $\mathbf f(\mathbf y^*) = 0$. La 1re équation donne
$X = Y$ ; la 3e, $Z = X^2/b$ ; la 2e s'écrit alors $X(r - 1 - Z) = 0$. Outre
l'origine $O$ (fluide au repos, conduction), il existe pour $r > 1$ deux points
fixes $C_\pm = (\pm\sqrt{b(r-1)}, \pm\sqrt{b(r-1)}, r-1)$ : les rouleaux de
convection stationnaires, tournant dans un sens ou dans l'autre
(`points_fixes(r, b)`).

La **jacobienne** est la matrice des dérivées partielles $J_{ij} = \partial f_i/\partial y_j$
(`jacobienne_lorenz`) :
$$ J = \begin{pmatrix} -P_r & P_r & 0 \\ r - Z & -1 & -X \\ Y & X & -b \end{pmatrix}. $$
Une petite perturbation d'un point fixe obéit à $\dot{\delta\mathbf y} \approx J(\mathbf y^*)\,\delta\mathbf y$ :
le point fixe est **stable** si toutes les valeurs propres de $J(\mathbf y^*)$
ont une partie réelle **négative** (les perturbations sont amorties), instable
si l'une d'elles a une partie réelle positive.

On appelle **bifurcation de Hopf** le passage d'une paire de valeurs propres
complexes conjuguées $\lambda = \mu \pm i\omega$ d'une partie réelle négative à
une partie réelle positive : le point fixe devient instable en se mettant à
osciller (à la pulsation $\omega$).

**Question 5.1.a.** Tracez, pour $r \in [0, 30]$, la plus grande partie réelle
des valeurs propres en $O$ et en $C_+$ (`np.linalg.eigvals`,
`valeurs_propres_C_plus`). Trouvez les valeurs de $r$ où elle change de signe
avec `scipy.optimize.brentq`, qui trouve un zéro d'une fonction continue $g$ sur
un intervalle $[a, b]$ tel que $g(a)$ et $g(b)$ soient de signes opposés.
Indiquez sur la figure les transitions conduction/convection et
convection/chaos (le chaos apparaît vers $r = 24.06$ ou $r = 24.74$ selon la
condition initiale, section 5.2).

In [ ]:
# À compléter : écrivez votre code ici

**Polynôme caractéristique.** Les valeurs propres sont les racines de
$\det(J - \lambda I) = 0$. En $C_\pm$, en développant selon la première ligne
(avec $X^2 = b(r-1)$ et $Z = r - 1$) :
$$ \lambda^3 + (P_r + b + 1)\lambda^2 + b(P_r + r)\lambda + 2P_r b(r-1) = 0. $$

**Question 5.1.b** (sur papier). Démontrez ce polynôme. À la bifurcation de
Hopf, $\lambda = i\omega$ est racine : remplacez $\lambda$ par $i\omega$, séparez
parties réelle et imaginaire, et retrouvez $\omega^2 = b(P_r + r)$ puis la
valeur du seuil $r_H = P_r(P_r + b + 3)/(P_r - b - 1)$. Comparez à la valeur
obtenue avec `brentq`.

**Fenêtres périodiques.** Au-delà de $r_H$, le régime est en général chaotique :
sur un diagramme de bifurcation (maxima successifs de $Z$ en fonction de $r$),
on voit alors un nuage de points. Mais il existe des intervalles de $r$, au
milieu de la zone chaotique, où la trajectoire finit par se répéter
exactement : elle a convergé vers une **orbite périodique** (un cycle limite).
Ces intervalles s'appellent des **fenêtres périodiques**. Ce n'est pas du
chaos : la suite des maxima de $Z$ n'y prend qu'un petit nombre de valeurs, et
le diagramme montre quelques branches nettes au lieu d'un nuage (par exemple
vers $r \approx 100$, pour $r \approx 145$ à 166, et pour tout $r > 313$).

### 5.2 Deux pièges : régime transitoire et bistabilité

**Question 5.2.a (régime transitoire).** Intégrez $r = 23$ depuis $(1,1,1)$ avec
RK4, $h = 0.01$, jusqu'à $T = 100$, et tracez $|Z(t) - (r-1)|$ (écart au point
fixe) en échelle log. La trajectoire semble d'abord chaotique (**régime
transitoire**), puis converge vers un point fixe. Peut-on savoir qu'elle se
stabilise sans calculer très loin en temps ? Oui : près de $C_\pm$, l'écart
décroît comme $e^{\mathrm{Re}\,\lambda\, t}$, où $\lambda$ est la valeur propre de
$C_+$ de plus grande partie réelle. Ajustez l'enveloppe de la fin de la courbe
(`trace_enveloppe` avec `tmin`), comparez son taux à $\mathrm{Re}\,\lambda$ et
superposez l'enveloppe théorique. Quel est le **temps caractéristique** de la
convergence ($1/|\mathrm{Re}\,\lambda|$) ? Combien de temps dure le régime
transitoire ? En quoi cela fausse-t-il un diagramme de bifurcation calculé avec
un transitoire trop court ?

In [ ]:
# À compléter : écrivez votre code ici

**Bistabilité.** Un système est **bistable** pour une valeur de $r$ si deux
régimes stables y coexistent : selon la condition initiale, la trajectoire
rejoint l'un ou l'autre.

**Question 5.2.b.** Pour $r = 24.5$ (entre 24.06 et $r_H$), intégrez jusqu'à
$T = 100$ depuis $C_+ + 10^{-2}(1,1,1)$, puis depuis $(1,1,1)$. Tracez les deux
trajectoires dans l'espace des phases (en 3D, `projection="3d"`). Quels sont les
deux régimes qui coexistent ?

In [ ]:
# À compléter : écrivez votre code ici

**Hystérésis.** On imite une expérience où l'on augmente puis diminue
lentement l'écart de température. `balayage_quasi_statique(rs, y0, Pr, b, bruit=...)`
fait varier $r$ par paliers : à chaque palier, elle part de l'état final du
palier précédent (le fluide « se souvient » de son histoire), ajoute une
perturbation aléatoire d'amplitude `bruit`, intègre pendant 200 et renvoie
l'amplitude des oscillations $A(r) = \max|Z(t) - (r-1)|$ sur la seconde moitié
du palier (petite sur les rouleaux stationnaires, $\sim 20$ en régime
chaotique). Si la montée et la descente ne donnent pas le même seuil, on dit
qu'il y a **hystérésis**.

**Question 5.2.c.** Faites une montée de $r = 22$ à 26 par pas de 0.1 en partant
de $C_+(22)$, puis une descente de 26 à 22 en partant de l'état final de la
montée, pour trois niveaux de bruit : 0, $10^{-3}$ et 1 (un panneau par niveau).
Relevez les seuils de chaos à la montée et à la descente et comparez-les à $r_H$
et à 24.06. Que fait un bruit fort ?

In [ ]:
rs_balayage = np.round(np.arange(22, 26.01, 0.1), 2)
# À compléter : écrivez votre code ici

### 5.3 Le schéma déplace-t-il les seuils ?

Près d'un point fixe, chaque mode propre est multiplié à chaque pas par
$R(h\lambda_i)$. Le point fixe est donc stable **pour le schéma** si et
seulement si $|R(h\lambda_i)| < 1$ pour tout $i$ : il perd sa stabilité quand
$\max_i |R(h\lambda_i)|$ franchit 1, ce qui n'a aucune raison de se produire au
même $r$ que pour le système exact (où c'est $\max_i \mathrm{Re}\,\lambda_i$ qui franchit 0).

**Question 5.3.a** (sur papier). Rappel (section 2.2) : un schéma est
**consistant** si son erreur locale divisée par $h$ tend vers 0 ; on montre que
cela équivaut à $R(z) = 1 + z + O(z^2)$ près de $z = 0$. Vérifiez-le pour
Euler explicite et pour Euler implicite (développez $1/(1-z)$). Montrez ensuite
que le seuil conduction/convection $r = 1$ est exactement préservé par tout
schéma consistant, quel que soit $h$. *Indications :* (1) les points fixes du
schéma sont ceux du système (section 4.2) ; (2) en $O$, quand $r$ passe par 1,
c'est une valeur propre **réelle** $\lambda$ qui passe de $\lambda < 0$ (stable)
à $\lambda > 0$ (instable) ; (3) pour $z = h\lambda$ réel proche de 0,
$R(z) \approx 1 + z$ est inférieur à 1 si $\lambda < 0$, supérieur à 1 si $\lambda > 0$.

**Question 5.3.b.** `seuil_hopf_numerique(nom_schema, h, Pr, b)` calcule par
`brentq` la valeur $r_H^{num}(h)$ où $C_+$ perd sa stabilité **pour le schéma**
(bifurcation de Hopf du schéma). Calculez-la pour les neuf schémas et
$h = 0.02, 0.01, 0.005, 0.0025, 0.00125$. Tracez $|r_H^{num} - r_H|$ en fonction de
$h$ et mesurez les pentes. Pourquoi Euler implicite n'a-t-il pas de seuil
pour $h \ge 0.01$ ? Pourquoi Crank-Nicolson et le point milieu ont-ils un
décalage nul ? Les schémas d'ordre $p$ pair (Heun, RK4, BDF2, SDIRK2, TR-BDF2)
ont une pente $p + 1$ et non $p$ : pourquoi ? *Indication :* pour un schéma
d'ordre $p$, $R(z) - e^z = c\,z^{p+1} + O(z^{p+2})$ ; calculez $|R(iy)|^2$ en
remarquant que $(iy)^{p+1}$ est imaginaire pur si $p$ est pair.

In [ ]:
hs_seuil = np.array([0.02, 0.01, 0.005, 0.0025, 0.00125])
# À compléter : écrivez votre code ici

**Question 5.3.c (vérification par simulation).** Partez de $C_+ + 10^{-4}(1,1,1)$
et intégrez pendant 30 : la perturbation oscille autour de $C_+$ avec une
amplitude qui croît ou décroît exponentiellement. `trace_enveloppe` mesure le
taux $\sigma$ de l'enveloppe de $Z(t) - (r-1)$. Pour chaque schéma, choisissez
deux valeurs de $r$ écartées de 1, de part et d'autre de son seuil numérique
$r_H^{num}(h)$ : RK4 avec $h = 0.01$ ($r = 24$ et 25), Euler explicite avec
$h = 0.01$ ($r = 15.5$ et 16.5), Euler implicite avec $h = 0.0025$ ($r = 29$ et 30).
Disposez les figures en colonnes (une par schéma). Comparez $\sigma$ au taux
prédit $\sigma_{num} = \max_i \frac1h\ln|R(h\lambda_i)|$ et au taux exact
$\max_i \mathrm{Re}\,\lambda_i$. Ce taux mesure l'énergie que le schéma apporte
aux oscillations ou leur retire.

In [ ]:
cas_enveloppe = [("RK4", 0.01, [24, 25]), ("Euler explicite", 0.01, [15.5, 16.5]),
                 ("Euler implicite", 0.0025, [29, 30])]
# À compléter : écrivez votre code ici

**Question 5.3.d (et le chaos ?).** Partez de $C_+ + 10^{-4}(1,1,1)$ et intégrez
jusqu'à $T = 200$ : $r = 20$ avec Euler explicite ($h = 0.01$), et $r = 28$ avec
Euler implicite ($h = 0.01$), en comparant chaque fois avec RK4. Tracez
$|Z(t) - (r-1)|$ (échelle log) et superposez les droites de croissance prévues
$10^{-4} e^{\sigma t}$ : $\sigma = \sigma_{num}$ du schéma et $\sigma = \mathrm{Re}\,\lambda$
(système exact). Tant que la perturbation est petite, elle suit la prévision ;
ensuite, les termes non linéaires prennent le relais. Interprétez
physiquement : pourquoi Euler explicite fait-il apparaître du chaos là où le
fluide devrait former des rouleaux stationnaires, et Euler implicite le
fait-il disparaître ? Quelle erreur commettrait-on sur l'écart de température
critique avec Euler explicite et $h = 0.01$ ?

In [ ]:
# À compléter : écrivez votre code ici

---
# 6. Conservation et raideur

### 6.1 Un système de Lorenz sans dissipation

Un **invariant** d'un système dynamique est une fonction $C(\mathbf y)$ qui ne
varie pas le long des trajectoires : $\frac{d}{dt} C(\mathbf y(t)) = 0$ (par
exemple l'énergie d'un système sans frottement). Dans le système de Lorenz,
les termes $-P_r X$ (viscosité), $-Y$ et $-bZ$ (diffusion thermique) dissipent
l'énergie. En les supprimant, on obtient le système sans dissipation
$\dot X = P_r Y$, $\dot Y = -XZ + rX$, $\dot Z = XY$ (`systeme_conservatif`).

**Question 6.1.a.** Montrez, en dérivant par rapport au temps et en utilisant
les équations, que $C_1 = Y^2 + Z^2 - 2rZ$ et $C_2 = X^2 - 2P_rZ$ sont des
invariants. Vérifiez-le numériquement avec RK4, $h = 10^{-3}$, $T = 10$
(`invariants_conservatif`).

In [ ]:
r = 28
f_cons = lambda t, y: systeme_conservatif(t, y, Pr, r)
jac_cons = lambda t, y: jacobienne_conservatif(t, y, Pr, r)
# À compléter : écrivez votre code ici

**Question 6.1.b.** $r = 28$, $\mathbf y_0 = (1, 0, 0)$, $T = 100$, $h = 0.01$ : pour
les neuf schémas, tracez le maximum sur $[0, t]$ de $|C_1(s) - C_1(0)|$
(`np.maximum.accumulate` calcule ce maximum cumulé, qui distingue une erreur
bornée d'une dérive). Utilisez une échelle « symlog »
(`ax.set_yscale("symlog", linthresh=1e-12)`), logarithmique au-dessus de $10^{-12}$
et linéaire autour de 0, pour pouvoir tracer la valeur 0. Classez les
comportements : conservation à l'arrondi près, erreur bornée, dérive qui croît
avec le temps, amortissement, explosion. Comment la dérive de RK4 dépend-elle
de $t$ et de $h$ (refaites RK4 avec $h = 0.005$) ? Reliez aux valeurs de $|R(iy)|$ (2.4).

In [ ]:
# À compléter : écrivez votre code ici

**Pourquoi le point milieu conserve-t-il exactement $C_1$ et $C_2$ ?** Ce sont
des fonctions **quadratiques** de $\mathbf y$. Pour toute fonction quadratique
$Q$, on a exactement $Q(\mathbf y_{n+1}) - Q(\mathbf y_n) = \nabla Q(\bar{\mathbf y})\cdot(\mathbf y_{n+1} - \mathbf y_n)$
avec $\bar{\mathbf y} = (\mathbf y_n + \mathbf y_{n+1})/2$ (vérifiez-le pour
$Q(y) = y^2$). Or le point milieu s'écrit $\mathbf y_{n+1} - \mathbf y_n = h\,\mathbf f(\bar{\mathbf y})$,
et $\nabla Q\cdot\mathbf f = dQ/dt = 0$ partout : $Q$ ne varie pas.
Crank-Nicolson moyenne les **valeurs** de $\mathbf f$ au lieu d'évaluer $\mathbf f$
au point moyen : les deux schémas coïncident pour un système linéaire (2.7),
mais pas ici. On admettra que Crank-Nicolson reste à une distance $O(h)$ du
point milieu, d'où une erreur bornée sur les invariants.

**Question 6.1.c** (interprétation physique). Un schéma qui ne conserve pas
$C_1$, $C_2$ ajoute ou retire l'équivalent d'une viscosité numérique. Dans le
système complet, que risque-t-on quand cette viscosité numérique est
comparable à la viscosité physique ? Faites le lien avec la section 5.3.

### 6.2 Un système de Lorenz raide : grand nombre de Prandtl et condition CFL

Les huiles silicones ont $P_r \sim 10^3$ à $10^4$ (le manteau terrestre,
$\sim 10^{23}$). On prend $P_r = 10^4$ et $r = 28$. Rappel : parmi les schémas
à pas fixe, Euler implicite, BDF2, SDIRK2 et TR-BDF2 sont **L-stables** ;
Crank-Nicolson et le point milieu sont A-stables mais **non L-stables** ;
Euler explicite, Heun et RK4 ne sont pas A-stables. Parmi les méthodes de
`solve_ivp`, `Radau` et `BDF` sont adaptées aux problèmes raides, `RK45` et
`DOP853` ne le sont pas.

**Question 6.2.a.** Calculez les valeurs propres de la jacobienne en $C_+$.
Quel est le rapport entre la plus rapide et la plus lente des échelles de
temps ? Pourquoi le problème est-il raide ? Prédisez, par la condition CFL
(2.5), le pas maximal d'Euler explicite et de RK4.

In [ ]:
Pr_raide = 1e4
f_raide = lambda t, y: systeme_dynamique(t, y, Pr_raide, 28, b)
jac_raide = lambda t, y: jacobienne_lorenz(t, y, Pr_raide, 28, b)
# À compléter : écrivez votre code ici

**Question 6.2.b (mesure de la condition CFL).** Pour Euler explicite puis RK4,
cherchez par **dichotomie** le plus grand pas stable : partez de
$[h_{min}, h_{max}] = [10^{-5}, 10^{-3}]$ ; prenez $h = \sqrt{h_{min}h_{max}}$ ;
intégrez sur $T = 1$ depuis $(10, 1, 1)$ avec `integration_temporelle_numba` ;
si la solution n'est pas finie ou dépasse $10^{10}$ (instabilité), remplacez
$h_{max}$ par $h$, sinon $h_{min}$ par $h$ ; répétez 20 fois. Comparez à la prédiction.

In [ ]:
# À compléter : écrivez votre code ici

**Question 6.2.c (modes parasites).** Avec $h = 0.01$ (50 fois le pas maximal
d'Euler explicite) et $\mathbf y_0 = (10, 1, 1)$ (donc $X \neq Y$), intégrez 20 pas
avec les neuf schémas et tracez $|X_n - Y_n|$ (échelle log). Physiquement,
$X - Y$ (le mode rapide) doit disparaître en un temps $\sim 1/P_r = 10^{-4}$, bien
moins qu'un pas. Qu'observe-t-on pour les schémas non A-stables ? pour
Crank-Nicolson et le point milieu (calculez $R(-10^4 h)$) ? pour les schémas L-stables ?

In [ ]:
# À compléter : écrivez votre code ici

**Question 6.2.d (coût, travail-précision pour un système raide).** On veut
$\mathbf y(10)$ à $10^{-4}$ près. La référence est calculée avec `Radau` et des
tolérances $10^{-12}$ : `DOP853`, méthode explicite, est soumise à la condition
CFL et devrait faire des centaines de milliers de pas, alors que `Radau`
(implicite, L-stable) suit la dynamique lente avec de grands pas. Comparez le
temps et l'erreur finale de : RK4 avec $h = 2.5\cdot10^{-4}$ (juste sous la limite de
stabilité), SDIRK2 et BDF2 avec $h = 0.01$, et `RK45`, `DOP853`, `Radau`, `BDF`,
`LSODA` de `solve_ivp` avec `rtol = atol = 1e-6`. Comparez au diagramme
travail-précision du système non raide (4.3) : qu'est-ce qui a changé ?

In [ ]:
# À compléter : écrivez votre code ici

### 6.3 Contraction des volumes

La **divergence** $\mathrm{div}\,\mathbf f = \partial f_1/\partial X + \partial f_2/\partial Y + \partial f_3/\partial Z$
(la trace de la jacobienne) mesure la variation des volumes de l'espace des
phases : un petit volume de conditions initiales est multiplié par
$e^{h\,\mathrm{div}\,\mathbf f}$ pendant un temps $h$. Pour un schéma, ce facteur
est le déterminant de la jacobienne $D$ de l'application
$\Phi_h : \mathbf y_n \mapsto \mathbf y_{n+1}$. Aucune bibliothèque ne fournit
directement ce déterminant pour une fonction quelconque : on estime $D$ par
différences finies centrées (colonne $j$ :
$(\Phi_h(\mathbf y + \eta\mathbf e_j) - \Phi_h(\mathbf y - \eta\mathbf e_j))/(2\eta)$,
$\eta = 10^{-6}$), puis on utilise `np.linalg.det`.

**Question 6.3.** Calculez $\mathrm{div}\,\mathbf f$ pour Lorenz. Pourquoi
l'attracteur ne peut-il pas remplir un volume ? Pour Euler explicite, Heun,
RK4, Crank-Nicolson et le point milieu, avec $h = 0.01$ et 0.005, calculez la
moyenne de $\frac1h \ln\det D$ sur 200 points de l'attracteur et comparez à
$\mathrm{div}\,\mathbf f$. Avec quel ordre l'écart décroît-il ?

In [ ]:
f28 = lambda t, y: systeme_dynamique(t, y, Pr, 28, b)
jac28 = lambda t, y: jacobienne_lorenz(t, y, Pr, 28, b)
points = points_sur_attracteur(200, Pr, 28, b)
# À compléter : écrivez votre code ici

---
# 7. Apprentissage automatique : interpoler n'est pas extrapoler

Les données sont fournies dans `data` (`donnees_apprentissage.py` montre comment
elles ont été calculées). On utilise `scikit-learn`. Deux règles : séparer
entraînement et test **par intervalles** de paramètres, pour distinguer
**interpolation** (test à l'intérieur du domaine d'entraînement) et
**extrapolation** (à l'extérieur) ; toujours comparer à une méthode simple.

### 7.1 Régression d'une loi connue

`charge_donnees("amplitudes")` contient l'amplitude $A(r) = |X|$ de la
convection stationnaire pour 200 valeurs de $r \in [1.05, 23]$.

**Question 7.1.** Entraînez sur $r \le 12$ privé de $[6, 8]$ ; testez sur $[6, 8]$
(interpolation) et $[12, 23]$ (extrapolation) : un polynôme de degré 5
(`np.polyfit`), une forêt aléatoire (`sklearn.ensemble.RandomForestRegressor`),
un réseau de neurones (`sklearn.neural_network.MLPRegressor`, précédé d'une
normalisation `sklearn.preprocessing.StandardScaler` dans un
`sklearn.pipeline.make_pipeline`, avec 3 couches de 50 neurones, l'activation
`tanh` et le solveur `lbfgs`, adapté aux petits jeux de données) et une régression linéaire de $A^2$ en fonction de $r$. Tracez les
données sur tout $[1.05, 23]$ et les prédictions ; donnez l'erreur maximale sur
chaque intervalle de test. Comment chaque modèle se comporte-t-il hors du
domaine d'entraînement, et pourquoi ? Que retenez-vous sur le rôle des
connaissances physiques en extrapolation ?

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from donnees_apprentissage import charge_donnees

donnees = charge_donnees("amplitudes")
rs_A = np.array(donnees["r"])
A = np.array(donnees["A"])
entrainement = (rs_A <= 12) & ~((rs_A >= 6) & (rs_A <= 8))
interpolation = (rs_A >= 6) & (rs_A <= 8)
extrapolation = rs_A > 12
# À compléter : écrivez votre code ici

### 7.2 Classifier des bassins d'attraction

Pour $1 < r < 24.06$, toute trajectoire finit sur $C_+$ ou $C_-$ selon la
condition initiale : l'ensemble des conditions initiales qui mènent à $C_+$
est le **bassin d'attraction** de $C_+$. `charge_donnees("bassins_r10")` et
`"bassins_r20"` contiennent 4000 conditions initiales $(X_0, Y_0) \in [-30, 30]^2$
(avec $Z_0 = r - 1$) et l'étiquette $+1$ ($C_+$) ou $-1$ ($C_-$) du point fixe atteint.

**Question 7.2.** Représentez les deux bassins. Entraînez un classifieur des
$k$ plus proches voisins (`sklearn.neighbors.KNeighborsClassifier`), une forêt
aléatoire (`sklearn.ensemble.RandomForestClassifier`) et un réseau de neurones
(`sklearn.neural_network.MLPClassifier`) sur les $N = 250, 500, 1000, 3000$
premiers points ; mesurez la précision (méthode `score`) sur les 1000 derniers ;
tracez les **courbes d'apprentissage** pour $r = 10$ et 20. Pourquoi ajouter des
données n'aide-t-il presque plus pour $r = 20$ ? Faites le lien avec le régime
transitoire chaotique (5.2).

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
# À compléter : écrivez votre code ici

### 7.3 Pour aller plus loin

Ces pistes ne sont pas comptées dans les 10 heures.
* **Redécouvrir les équations à partir des données** (méthode SINDy). On
  cherche $\dot{\mathbf y} \approx \Theta(\mathbf y)\,\Xi$, où $\Theta$ contient les
  10 monômes de degré au plus 2 en $(X, Y, Z)$ (`sklearn.preprocessing.PolynomialFeatures`)
  et $\Xi$ une matrice de coefficients presque tous nuls. Estimez $\dot{\mathbf y}$
  à partir d'une trajectoire échantillonnée (`np.gradient`), puis calculez $\Xi$
  par régression parcimonieuse (`sklearn.linear_model.Lasso`, ou moindres carrés
  `np.linalg.lstsq` répétés en annulant les petits coefficients). Retrouve-t-on
  $P_r = 10$, $r = 28$, $b = 8/3$ ? Que se passe-t-il si l'on ajoute du bruit aux données ?
* **Prévoir une trajectoire chaotique** avec un réseau à réservoir (*echo state
  network*) : un réseau récurrent aléatoire de quelques centaines de neurones,
  dont seule la sortie linéaire est entraînée (`sklearn.linear_model.Ridge`).
  Combien de temps de Lyapunov la prévision reste-t-elle valable ? Comparez à
  l'horizon de prédictibilité de la section 4.4.
* **Réseaux informés par la physique** (PINN) : un réseau de neurones représente
  $t \mapsto \mathbf y(t)$ et l'on minimise le résidu $\dot{\mathbf y} - \mathbf f(\mathbf y)$.
  Cela demande une bibliothèque de différentiation automatique (PyTorch ou
  JAX, non incluses dans `requirements.txt`). Sur Lorenz, l'entraînement
  échoue dès que l'intervalle de temps dépasse quelques unités : pourquoi ?